# Anomaly detection of a thyratron grid waveform

Copyright (C) 2024 Hirokazu Maesaka, RIKEN SPring-8 Center. All rights reserved.

This program is free software: you can redistribute it and/or modify it under the terms of the GNU General Public License as published by the Free Software Foundation, either version 3 of the License, or (at your option) any later version.

This program is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY; without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. See the GNU General Public License for more details.

You should have received a copy of the GNU General Public License along with this program. If not, see <https://www.gnu.org/licenses/>.

In [ ]:
import re
import tarfile
import urllib.request
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
plt.rcParams['font.size'] = 16
%matplotlib inline

## Get the data
The waveform files are expected in the directory `data/`. If it does not exist, the archive `wfmdata.tar.gz` is
extracted; if the archive does not exist either (e.g. in Google Colaboratory), it is downloaded from the GitHub
repository of this lecture.

In [ ]:
DATA_DIR = Path("data")
ARCHIVE = Path("wfmdata.tar.gz")
ARCHIVE_URL = "https://raw.githubusercontent.com/e362scibar/isba-examples/main/wfmdata.tar.gz"

In [ ]:
if not DATA_DIR.is_dir():
    if not ARCHIVE.is_file():
        print(f"Downloading {ARCHIVE_URL}")
        urllib.request.urlretrieve(ARCHIVE_URL, ARCHIVE)
    print(f"Extracting {ARCHIVE}")
    with tarfile.open(ARCHIVE, "r:gz") as archive:
        archive.extractall(".", filter="data")

# Classification of input data
Each file `sn<serial>_<hours>h_g1.npz` contains about 100 grid waveforms (160 samples each) of one tube with the
serial number `<serial>` after `<hours>` hours of operation. Two tubes also have a second file `_g2.npz`, which is
not used here.

The tubes are classified into three classes:

| Class | Operating hours | Meaning |
|---|---|---|
| New | < 5000 h | new tubes |
| Aged | 5000–20000 h | tubes in operation |
| Failed | ≥ 20000 h | tubes that actually failed |

In this dataset, the failed tubes are exactly those with at least 20000 hours, so the class is assigned from the
operating hours in the file name. In general, the labels must come from the maintenance records.

In [ ]:
NEW_LIMIT_HOURS = 5000
FAILED_LIMIT_HOURS = 20000
CLASS_NAMES = ["new", "aged", "failed"]
NEW, AGED, FAILED = range(3)
FILE_PATTERN = re.compile(r"sn(\d+)_(\d+)h_g1\.npz")


def tube_class(hours: int) -> int:
    """Return the class index of a tube from its operating hours."""
    if hours < NEW_LIMIT_HOURS:
        return NEW
    if hours < FAILED_LIMIT_HOURS:
        return AGED
    return FAILED

In [ ]:
thyIdList = [[], [], []]  # (serial, hours) per class: new, aged, failed
dfList = [[], [], []]  # waveform arrays per class, shape (number of waveforms, samples)
for path in sorted(DATA_DIR.glob("sn*_g1.npz")):
    serial, hours = map(int, FILE_PATTERN.fullmatch(path.name).groups())
    with np.load(path) as npz:
        waveforms = npz["wfm"]
    class_index = tube_class(hours)
    thyIdList[class_index].append((serial, hours))
    dfList[class_index].append(waveforms)

for class_index, name in enumerate(CLASS_NAMES):
    print(f"{name}: {len(thyIdList[class_index])} tubes")

# S/N 2176 (18100 h) seems to be in good condition (see the regression results below)

## Training data
The PCA is fitted to the waveforms of the new and failed tubes. The aged tubes are kept for testing.

In [ ]:
tdf = np.vstack([waveforms for class_index in (NEW, FAILED) for waveforms in dfList[class_index]])
print(f"Training waveforms: {tdf.shape}")

# PCA
The PCA subtracts the mean waveform and decomposes the centered data matrix by the singular value decomposition
$M - \bar M = U\Sigma V^\top$. The rows of $V^\top$ (`pca.components_`) are the waveform shapes of the principal
components, and the principal-component scores of the waveforms are the rows of $U\Sigma$ (`pca.transform`).
The contribution ratio of the $i$-th component is $\sigma_i^2/\sum_j\sigma_j^2$.

In [ ]:
N_COMPONENTS = 6
pca = PCA(n_components=N_COMPONENTS)
pca.fit(tdf)

## Cumulative contribution ratio

In [ ]:
cumulative_ratio = np.insert(np.cumsum(pca.explained_variance_ratio_), 0, 0.)
fig, ax = plt.subplots()
ax.plot(cumulative_ratio, 'o-')
ax.set_ylim(0., 1.05)
ax.grid()
ax.set_xlabel('Number of principal components')
ax.set_ylabel('Cumulative contribution ratio')
plt.show()
print("Cumulative contribution ratio:", np.round(cumulative_ratio[1:], 3))

## Plot the PCA outputs of the first 10 tubes of each class

In [ ]:
N_PLOT_TUBES = 10
MARKERS = ['+', 'x', '.']
fig, ax = plt.subplots(figsize=[8, 8])
for class_index, waveform_list in enumerate(dfList):
    for (serial, hours), waveforms in zip(thyIdList[class_index][:N_PLOT_TUBES], waveform_list):
        scores = pca.transform(waveforms)
        ax.scatter(scores[:, 0], scores[:, 1], marker=MARKERS[class_index], label=f'SN: {serial} ({hours}hr)')
ax.set_xlabel('PCA-1')
ax.set_ylabel('PCA-2')
ax.grid()
ax.legend(loc='center left', bbox_to_anchor=(1., 0.5), fontsize=9)
plt.show()

## Plot the PCA outputs of the training data (new and failed tubes)

In [ ]:
fig, ax = plt.subplots(figsize=[8, 8])
for class_index in (NEW, FAILED):
    for (serial, hours), waveforms in zip(thyIdList[class_index][:N_PLOT_TUBES], dfList[class_index]):
        scores = pca.transform(waveforms)
        ax.scatter(scores[:, 0], scores[:, 1], marker='+' if class_index == NEW else '.', label=f'SN: {serial} ({hours}hr)')
ax.set_xlabel('PCA-1')
ax.set_ylabel('PCA-2')
ax.grid()
ax.legend(loc='center left', bbox_to_anchor=(1., 0.5), fontsize=9)
plt.show()

## Waveform of each component

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 7), constrained_layout=True)
for component_index, ax in enumerate(axes.flat):
    ax.plot(pca.components_[component_index, :])
    ax.grid()
    ax.set_title(f'Component {component_index+1}')
    ax.set_xlabel('Sample index')
plt.show()

## Linear regression
The principal-component scores $x_i$ ($i = 1, \dots, N$) of each waveform are fed into a linear regression
$$ y = \sum_{i=1}^{N} w_i x_i + b, $$
trained with $y = 0$ for the new tubes and $y = 1$ for the failed tubes. The aged tubes are used as a test dataset.
(A linear regression on 0/1 labels is a *linear probability model*; a logistic regression is the more common choice
for classification.)

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
# Select new and failed tubes for training and aged tubes for testing
xtrain = np.vstack([pca.transform(waveforms) for class_index in (NEW, FAILED) for waveforms in dfList[class_index]])
ytrain = np.hstack([
    np.full(len(waveforms), float(class_index == FAILED)) for class_index in (NEW, FAILED) for waveforms in dfList[class_index]
])
xtest = np.vstack([pca.transform(waveforms) for waveforms in dfList[AGED]])
print(xtrain.shape, ytrain.shape)
print(xtest.shape)

In [ ]:
# Fit
SCORE_RANGE = (-0.5, 1.5)
N_BINS = 100


def plot_score_histograms(model, title: str) -> None:
    """Histogram the regression output for the training (new, failed) and test (aged) waveforms."""
    fig, ax = plt.subplots()
    ax.hist(model.predict(xtrain[ytrain == 0]), N_BINS, range=SCORE_RANGE, alpha=0.8, label='train new')
    ax.hist(model.predict(xtrain[ytrain == 1]), N_BINS, range=SCORE_RANGE, alpha=0.8, label='train failed')
    ax.hist(model.predict(xtest), N_BINS, range=SCORE_RANGE, alpha=0.8, label='test aged')
    ax.grid()
    ax.legend()
    ax.set_title(title)
    ax.set_xlabel('Regression output')
    plt.show()


linear_model = LinearRegression().fit(xtrain, ytrain)
plot_score_histograms(linear_model, 'Linear regression')

## Support vector machine (SVM) regression
`SVR` (support vector regression) with the RBF kernel
$K(\boldsymbol{x}, \boldsymbol{x}') = \exp(-\gamma\|\boldsymbol{x}-\boldsymbol{x}'\|^2)$ fits a function that may deviate
from the target by at most $\epsilon$ without penalty, while keeping the function as flat as possible.
Errors larger than $\epsilon$ are penalized linearly, and the fitted function is determined by the training points
on or outside the $\epsilon$ tube, the *support vectors*. (The margin maximization between classes is the idea of the
SVM *classifier*, `SVC`.)

The principal-component scores are used without standardization, so the RBF kernel is dominated by the first
components, which have the largest variance. This works well here; standardizing the scores
(`sklearn.preprocessing.StandardScaler`) would give all components the same weight.

In [ ]:
from sklearn.svm import SVR

In [ ]:
# Support vector machine regressor
svr_model = SVR()
svr_model.fit(xtrain, ytrain)

In [ ]:
plot_score_histograms(svr_model, 'SVM regression')

## Evaluation on unseen tubes (leave-one-tube-out)
The histograms above show the training waveforms themselves, so they do not tell how well the method works for a
tube that it has never seen. Moreover, the about 100 waveforms of one tube are very similar to each other, so
splitting the waveforms randomly would also leak information. Therefore, each new or failed tube is held out in
turn, the PCA and the regression are trained on the other tubes, and the held-out tube is evaluated.
A tube is classified as failed if the median output of its waveforms exceeds 0.5.

In [ ]:
DECISION_THRESHOLD = 0.5
training_tubes = [
    (class_index, tube_id, waveforms)
    for class_index in (NEW, FAILED)
    for tube_id, waveforms in zip(thyIdList[class_index], dfList[class_index])
]


def leave_one_tube_out(model_factory) -> list[tuple[int, tuple[int, int], np.ndarray]]:
    """Return (class, tube id, outputs of the held-out waveforms) for every held-out new or failed tube."""
    results = []
    for held_out in range(len(training_tubes)):
        rest = [tube for index, tube in enumerate(training_tubes) if index != held_out]
        x_rest = np.vstack([waveforms for _, _, waveforms in rest])
        y_rest = np.hstack([np.full(len(waveforms), float(class_index == FAILED)) for class_index, _, waveforms in rest])
        fold_pca = PCA(n_components=N_COMPONENTS).fit(x_rest)
        model = model_factory().fit(fold_pca.transform(x_rest), y_rest)
        class_index, tube_id, waveforms = training_tubes[held_out]
        results.append((class_index, tube_id, model.predict(fold_pca.transform(waveforms))))
    return results


fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, (name, factory) in zip(axes, [("Linear regression", LinearRegression), ("SVM regression", SVR)]):
    results = leave_one_tube_out(factory)
    wrong_tubes = [
        tube_id for class_index, tube_id, outputs in results
        if (np.median(outputs) > DECISION_THRESHOLD) != (class_index == FAILED)
    ]
    wrong_waveforms = sum(int(np.sum((outputs > DECISION_THRESHOLD) != (class_index == FAILED))) for class_index, _, outputs in results)
    total_waveforms = sum(len(outputs) for _, _, outputs in results)
    print(f"{name}: misclassified tubes {len(wrong_tubes)}/{len(results)} {wrong_tubes}, "
          f"misclassified waveforms {wrong_waveforms}/{total_waveforms}")
    for class_index, label in [(NEW, 'held-out new'), (FAILED, 'held-out failed')]:
        outputs = np.hstack([out for c, _, out in results if c == class_index])
        ax.hist(outputs, N_BINS, range=SCORE_RANGE, alpha=0.8, label=label)
    ax.axvline(DECISION_THRESHOLD, color='k', linestyle='--')
    ax.grid()
    ax.legend()
    ax.set_title(f'{name}, leave-one-tube-out', fontsize=14)
    ax.set_xlabel('Regression output')
fig.tight_layout()
plt.show()

## Aged tubes versus operating hours
The output of the model trained on all new and failed tubes is shown for each aged tube (median and interquartile
range of its waveforms) against the operating hours. A tube whose output approaches 1 has a waveform similar to
that of the failed tubes and is a candidate for preventive replacement.

In [ ]:
from scipy.stats import spearmanr

aged_hours = np.array([hours for _, hours in thyIdList[AGED]])
fig, ax = plt.subplots(figsize=(8, 5))
for name, model, offset in [("Linear regression", linear_model, -60.), ("SVM regression", svr_model, 60.)]:
    outputs = [model.predict(pca.transform(waveforms)) for waveforms in dfList[AGED]]
    medians = np.array([np.median(out) for out in outputs])
    lower = medians - np.array([np.percentile(out, 25) for out in outputs])
    upper = np.array([np.percentile(out, 75) for out in outputs]) - medians
    ax.errorbar(aged_hours + offset, medians, yerr=[lower, upper], fmt='o', capsize=3, label=name)
    rho, p_value = spearmanr(aged_hours, medians)
    print(f"{name}: Spearman rank correlation with the operating hours rho = {rho:.2f} (p = {p_value:.2f})")
ax.axhline(DECISION_THRESHOLD, color='k', linestyle='--')
ax.set_xlabel('Operating hours')
ax.set_ylabel('Regression output (median, IQR)')
ax.grid()
ax.legend()
plt.show()

- New and failed tubes are separated also for unseen tubes (leave-one-tube-out above).
- The aged tubes lie in between. Their output tends to increase with the operating hours, but the correlation is
  weak: some tubes with fewer hours are already close to the failed tubes, and some old tubes (e.g. S/N 2176 at
  18100 h) still look new. The waveform carries information on the condition of each tube that the operating hours
  alone do not provide.